In [1]:
import torch
import torch.nn as nn
import torchvision.transforms.functional as func

from torchvision.models import resnet18

class ResNet18Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = resnet18(weights=None)
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.pool = resnet.maxpool
        self.encoder1 = resnet.layer1
        self.encoder2 = resnet.layer2
        self.encoder3 = resnet.layer3
        self.encoder4 = resnet.layer4
    def forward(self, x):
        x0 = self.stem(x)
        x1 = self.encoder1(self.pool(x0))
        x2 = self.encoder2(x1)
        x3 = self.encoder3(x2)
        x4 = self.encoder4(x3)
        return x0, x1, x2, x3, x4


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(skip_channels + out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = self.up(x)
        skip = func.center_crop(skip, [x.shape[2], x.shape[3]])
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class ResNetUNet(nn.Module):
    def __init__(self, n_classes=1):
        super().__init__()
        self.encoder = ResNet18Encoder()

        self.decoder4 = DecoderBlock(512, 256, 256)
        self.decoder3 = DecoderBlock(256, 128, 128)
        self.decoder2 = DecoderBlock(128, 64, 64)
        self.decoder1 = DecoderBlock(64, 64, 32)

        self.final_conv = nn.Conv2d(32, n_classes, kernel_size=1)

    def forward(self, x):
        x0, x1, x2, x3, x4 = self.encoder(x)

        d4 = self.decoder4(x4, x3)
        d3 = self.decoder3(d4, x2)
        d2 = self.decoder2(d3, x1)
        d1 = self.decoder1(d2, x0)

        out = self.final_conv(d1)
        return out

In [2]:
from torch.utils.data import Dataset
import tifffile as tif
import random
from torchvision import transforms
import os

random.seed(42)

imageTransform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
maskTransform = transforms.Compose([transforms.ToTensor()])

class LandslideDataset(Dataset):
    def __init__(self, img_list, mask_list):
        self.img_list = img_list
        self.mask_list = mask_list
    
    def __len__(self):
        return len(self.img_list)
    
    def __getitem__(self, index):
        img_dir = self.img_list[index]
        mask_dir = self.mask_list[index]

        img = tif.imread(img_dir)
        mask = tif.imread(mask_dir)

        img = imageTransform(img)
        mask = (maskTransform(mask) > 0).float()

        return img, mask

path = "../../data/"
limit = 100
regions = [f for f in os.listdir(path) if (os.path.isdir(os.path.join(path, f)) and f != "study areas shp")]

regions_dict = dict()
regions_pos_weight = dict()

for region in regions:
    dataset_dir = path + region
    image_dir = os.path.join(dataset_dir, "img")
    img_list = os.listdir(image_dir)

    all_images = sorted(os.path.join(image_dir, f) for f in img_list)

    if len(all_images) > limit:
        all_images = random.sample(all_images, limit)

    regions_dict[region] = all_images
    regions_pos_weight[region] = 0

def estimate_pos_weight(mask_paths):
    pos, tot = 0, 0
    for path in mask_paths:
        m = tif.imread(path)
        m = torch.from_numpy((m > 0).astype("float32"))
        pos += m.sum().item()
        tot += m.numel()
    neg = max(tot - pos, 1.0)
    pos = max(pos, 1.0)
    return neg / pos

# for region in regions_dict:
#     masks = [f.replace("img", "mask") for f in regions_dict[region]]
    
#     regions_pos_weight[region] = estimate_pos_weight(masks)

In [3]:
import numpy as np
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### TUNE THIS CONSTANT
threshold = 0.6

def getMetrics(TP, TN, FP, FN):
    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
        
    iou_1  = TP / (TP + FP + FN + 1e-8)
    iou_0 = TN / (TN + FP + FN + 1e-8)
    miou = (iou_0 + iou_1) / 2 
        
    oa = (TP + TN)/(TP + TN + FP + FN + 1e-8)
    
    return precision, recall, f1, iou_1, miou, oa

def dice_loss(pred, target, smooth=1.):
    pred = torch.sigmoid(pred)
    pred_flat = pred.view(-1)
    target_flat = target.view(-1)
    intersection = (pred_flat * target_flat).sum()
    return 1 - ((2. * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth))

def boundary_f1_score(pred_mask, true_mask, tolerance=3):
    if isinstance(pred_mask, torch.Tensor):
        pred = pred_mask.detach().cpu().float()
    else:
        pred = torch.from_numpy(pred_mask).float()

    if isinstance(true_mask, torch.Tensor):
        true = true_mask.detach().cpu().float()
    else:
        true = torch.from_numpy(true_mask).float()

    pred = (pred > threshold).float()
    true = (true > threshold).float()

    def _to_bchw(m):
        if m.ndim == 2:
            m = m.unsqueeze(0).unsqueeze(0)
        elif m.ndim == 3:
            m = m.unsqueeze(0)
        return m

    pred = _to_bchw(pred)
    true = _to_bchw(true)

    def get_edges(mask):
        dil = F.max_pool2d(mask, kernel_size=3, stride=1, padding=1)
        ero = 1.0 - F.max_pool2d(1.0 - mask, kernel_size=3, stride=1, padding=1)
        edge = (dil - ero > 0).float()
        return edge

    edge_pred = get_edges(pred)
    edge_true = get_edges(true)

    if edge_pred.sum() == 0 and edge_true.sum() == 0:
        return 1.0
    if edge_pred.sum() == 0 or edge_true.sum() == 0:
        return 0.0

    if tolerance > 0:
        k = 2 * tolerance + 1
        pad = tolerance
        edge_true_dil = F.max_pool2d(edge_true, kernel_size=k, stride=1, padding=pad)
        edge_pred_dil = F.max_pool2d(edge_pred, kernel_size=k, stride=1, padding=pad)
    else:
        edge_true_dil = edge_true
        edge_pred_dil = edge_pred

    precision_b = (edge_pred * edge_true_dil).sum() / (edge_pred.sum() + 1e-8)
    recall_b = (edge_true * edge_pred_dil).sum() / (edge_true.sum() + 1e-8)

    if precision_b + recall_b == 0:
        return 0.0

    bf1 = 2.0 * precision_b * recall_b / (precision_b + recall_b)
    return bf1.item()

### TUNE THESE CONSTANTS
pos_weight = 4.5
lambda_dice = 1.0

criterion_ce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))

def seg_loss(logits, targets):
    return criterion_ce(logits, targets) + lambda_dice * dice_loss(logits, targets)

In [4]:
def train_model(model, optimizer, train_loader, val_loader, epochs, model_path, region_name):
    best_iou = 0.0
    patience = 10
    counter = 0

    print(f'region, epoch, train_loss, val_loss, precision, recall, f1, iou, miou, oa, bf1')

    for epoch in range(epochs):
        model.train()
        running_loss, n_samples = 0.0, 0

        for images, masks in train_loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True).float()

            optimizer.zero_grad(set_to_none=True)

            logits = model(images)
            if logits.shape[-2:] != masks.shape[-2:]:
                logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

            loss = seg_loss(logits, masks)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * masks.numel()
            n_samples += masks.numel()
        
        training_loss = running_loss / max(1, n_samples)

        model.eval()
        val_loss, val_num = 0.0, 0
        
        TP, FP, FN, TN = 0,0,0,0

        bf1_sum = 0.0
        bf1_count = 0

        with torch.no_grad():
            for images, masks in val_loader:
                images = images.to(device, non_blocking=True)
                masks = masks.to(device, non_blocking=True).float()

                logits = model(images)
                if logits.shape[-2:] != masks.shape[-2:]:
                    logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

                loss = seg_loss(logits, masks)
                val_loss += loss.item()

                preds = torch.sigmoid(logits) > threshold

                for i in range(preds.size(0)):
                    bf1 = boundary_f1_score(preds[i, 0], masks[i, 0], tolerance=3)
                    bf1_sum += bf1
                    bf1_count += 1

                preds_flat = preds.view(-1)
                mask_flat = masks.view(-1)

                TP += ((preds_flat == 1) & (mask_flat == 1)).sum().item()
                FP += ((preds_flat == 1) & (mask_flat == 0)).sum().item()
                FN += ((preds_flat == 0) & (mask_flat == 1)).sum().item()
                TN += ((preds_flat == 0) & (mask_flat == 0)).sum().item()
                
                val_num += 1
             
        avg_bf1 = bf1_sum / max(bf1_count, 1)
        precision, recall, f1, iou, miou, oa = getMetrics(TP, TN, FP, FN)

        print(f'{region_name}, {epoch+1}, {training_loss :.3f}, {val_loss / val_num :.3f}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}, {avg_bf1:.4f}')

        if epoch == 0:
            torch.save(model.state_dict(), model_path)

        if iou > best_iou:
            best_iou = iou
            counter = 0
            
            torch.save(model.state_dict(), model_path)
        elif iou != 0.0:
            counter += 1
            if counter >= patience:
                break

def test_model(model, test_loader):
    TP, FP, FN, TN = 0,0,0,0

    model.eval()

    bf1_sum = 0.0
    bf1_count = 0

    with torch.no_grad():
        for images, masks in test_loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True).float()

            logits = model(images)
            if logits.shape[-2:] != masks.shape[-2:]:
                logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

            preds = torch.sigmoid(logits) > threshold

            for i in range(preds.size(0)):
                bf1 = boundary_f1_score(preds[i, 0], masks[i, 0], tolerance=3)
                bf1_sum += bf1
                bf1_count += 1

            preds_flat = preds.view(-1)
            mask_flat = masks.view(-1)

            TP += ((preds_flat == 1) & (mask_flat == 1)).sum().item()
            FP += ((preds_flat == 1) & (mask_flat == 0)).sum().item()
            FN += ((preds_flat == 0) & (mask_flat == 1)).sum().item()
            TN += ((preds_flat == 0) & (mask_flat == 0)).sum().item()

    avg_bf1 = bf1_sum / max(bf1_count, 1)
    precision, recall, f1, iou, miou, oa = getMetrics(TP, TN, FP, FN)

    return precision, recall, f1, iou, miou, oa, avg_bf1

In [8]:
""" SOURCE ONLY """

from torch.utils.data import DataLoader
import torch.optim as optim
from sklearn.model_selection import train_test_split

batch_size = 16

output = "source_region,target_region,precision,recall,f1,iou,miou,oa,bf1"

for source_region in regions_dict:
    image_paths = regions_dict[source_region]
    mask_paths = [f.replace("img", "mask") for f in image_paths]

    train_img, val_img, train_mask, val_mask = train_test_split(
        image_paths, mask_paths, test_size=.2, random_state=42
    )

    train_dataset = LandslideDataset(train_img, train_mask)
    val_dataset = LandslideDataset(val_img, val_mask)

    trainLoader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
    valLoader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

    epochs = 40

    model = ResNetUNet()
    model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

    model_path = "../../models/baseline/final/" + source_region + ".pth"

    train_model(model, optimizer, trainLoader, valLoader, epochs, model_path, source_region)

    model.load_state_dict(torch.load(model_path, map_location=device))

    for target_region in regions_dict:
        if target_region != source_region:
            image_paths = regions_dict[target_region]
            mask_paths = [f.replace("img", "mask") for f in image_paths]

            test_dataset = LandslideDataset(image_paths, mask_paths)
            testLoader = DataLoader(test_dataset, batch_size=batch_size,shuffle=False,num_workers=0, pin_memory=torch.cuda.is_available())

            precision, recall, f1, iou, miou, oa, bf1 = test_model(model, testLoader)
            result = f'{source_region}, {target_region}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}, {bf1:.4f}'
            print(result)
            output += f'\n{result}'
    
with open("../../results/baseline/final_source_only_100_0.csv", "w") as f:
    f.write(output)

region, epoch, train_loss, val_loss, precision, recall, f1, iou, miou, oa, bf1
Hokkaido Iburi-Tobu, 1, 1.788, 1.763, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372, 0.0000
Hokkaido Iburi-Tobu, 2, 1.767, 1.757, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372, 0.0000
Hokkaido Iburi-Tobu, 3, 1.738, 1.746, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372, 0.0000
Hokkaido Iburi-Tobu, 4, 1.696, 1.724, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372, 0.0000
Hokkaido Iburi-Tobu, 5, 1.621, 1.672, 0.9348, 0.0450, 0.0859, 0.0449, 0.4923, 0.9398, 0.0212
Hokkaido Iburi-Tobu, 6, 1.485, 1.600, 0.7037, 0.2835, 0.4041, 0.2532, 0.5999, 0.9475, 0.1889
Hokkaido Iburi-Tobu, 7, 1.299, 1.432, 0.8109, 0.4152, 0.5492, 0.3785, 0.6673, 0.9572, 0.1794
Hokkaido Iburi-Tobu, 8, 1.134, 1.306, 0.7313, 0.5503, 0.6280, 0.4577, 0.7077, 0.9590, 0.2590
Hokkaido Iburi-Tobu, 9, 0.985, 1.142, 0.7218, 0.6233, 0.6689, 0.5025, 0.7311, 0.9612, 0.3206
Hokkaido Iburi-Tobu, 10, 0.895, 1.024, 0.7139, 0.6680, 0.6902, 0.5269, 0.7438, 0.962

In [ ]:
""" TARGET ONLY """

from torch.utils.data import DataLoader
import torch.optim as optim
from sklearn.model_selection import train_test_split

batch_size = 16

output = "target_region,precision,recall,f1,iou,miou,oa,bf1"

for target_region in regions_dict:
    image_paths = regions_dict[target_region]
    mask_paths = [f.replace("img", "mask") for f in image_paths]

    train_img, temp_img, train_mask, temp_mask = train_test_split(
        image_paths, mask_paths, test_size=.3, random_state=42
    )
    
    val_img, test_img, val_mask, test_mask = train_test_split(
        temp_img, temp_mask, test_size=.5, random_state=42
    )

    train_dataset = LandslideDataset(train_img, train_mask)
    val_dataset = LandslideDataset(val_img, val_mask)
    test_dataset = LandslideDataset(test_img, test_mask)

    trainLoader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
    valLoader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
    testLoader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

    epochs = 40

    model = ResNetUNet()
    model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

    model_path = "../../models/baseline/final/" + target_region + ".pth"

    # train_model(model, optimizer, trainLoader, valLoader, epochs, model_path, target_region)

    model.load_state_dict(torch.load(model_path, map_location=device))
    
    precision, recall, f1, iou, miou, oa, bf1 = test_model(model, testLoader)
    result = f'{target_region}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}, {bf1:.4f}'
    print(result)
    output += f'\n{result}'
    
with open("../../results/baseline/final_target_only.csv", "w") as f:
    f.write(output)

Hokkaido Iburi-Tobu, 0.7696, 0.7846, 0.7770, 0.6353, 0.7967, 0.9610, 0.5487
Jiuzhai valley (UAV-0.2m), 0.8480, 0.7755, 0.8101, 0.6809, 0.7883, 0.9148, 0.2303
Jiuzhai valley (UAV-0.5m), 0.5721, 0.7720, 0.6572, 0.4894, 0.7041, 0.9246, 0.3703
Lombok, 0.3647, 0.6384, 0.4642, 0.3023, 0.6343, 0.9668, 0.3708
Longxi River (SAT), 0.6099, 0.7528, 0.6739, 0.5082, 0.7296, 0.9535, 0.1702
Longxi River (UAV), 0.6987, 0.7301, 0.7140, 0.5552, 0.7235, 0.9047, 0.3854
Mengdong Township, 0.7237, 0.8290, 0.7728, 0.6297, 0.7781, 0.9347, 0.3573
Moxi town (UAV-0.2m), 0.8813, 0.8074, 0.8427, 0.7282, 0.8065, 0.9119, 0.1282
Moxi town (UAV-1m), 0.7922, 0.8370, 0.8140, 0.6864, 0.8309, 0.9766, 0.5983
Moxitaidi (SAT), 0.8279, 0.7776, 0.8020, 0.6694, 0.8205, 0.9731, 0.2814
Moxitaidi (UAV-0.6m), 0.5899, 0.6811, 0.6322, 0.4622, 0.6840, 0.9129, 0.2686
Moxitaidi (UAV-1m), 0.4276, 0.5419, 0.4780, 0.3141, 0.6337, 0.9542, 0.2377
palu, 0.5238, 0.4617, 0.4908, 0.3252, 0.6528, 0.9806, 0.4563
Tiburon Peninsula (planet), 0.7292, 

: 